[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_32_lora_pure.ipynb)

# 🟡 Medium: LoRA without Flax

*Training*
Problem 26's LoRA adapter with no Flax.

### Signature
```python
class LoRALinear:
    def __init__(self, in_features, out_features, rank, alpha=1.0, *, key): ...
    def __call__(self, x): ...        # (..., in_features) -> (..., out_features)
```

| | shape |
|---|---|
| `self.linear` | a `Linear(in_features, out_features)` — the frozen base |
| `self.lora_A` | `(in_features, rank)`, random, scaled by `0.01` |
| `self.lora_B` | `(rank, out_features)`, **zeros** |
| `self.scaling` | `alpha / rank` |

$$y = Wx + b + \frac{\alpha}{r}\,(x A) B$$

### B starts at zero, and that is the whole trick
`B = 0` makes `A @ B` zero, so at step 0 the adapter is an **exact no-op** and
the model behaves exactly as it did before you attached it. Fine-tuning then
moves away from the original smoothly instead of jolting it.

Initialising **both** to zero would be worse than useless — the gradient of a
product where both factors are zero is zero, so nothing would ever learn. One
random, one zero.

### Freezing gets easier, not harder
This is the one place where dropping Flax **simplifies** things. Problem 26
had to demote the base parameters to plain `nnx.Variable` so that
`nnx.state(self, nnx.Param)` — what an optimizer filters on — would see only
the adapter:

```python
self.linear.kernel = nnx.Variable(self.linear.kernel[...])   # problem 26
```

Here there is nothing to demote. You decide what to differentiate by choosing
what to pass to `jax.grad`, so "frozen" just means "not in the argument":

```python
jax.grad(loss)(A, B)      # the base never enters
```

### Why rank matters
The adapter costs `r · (in + out)` parameters instead of `in · out`. For a
4096x4096 projection at `r = 8` that is 65k instead of 16.7M — 0.4%. `alpha /
rank` keeps the update magnitude roughly constant as you change `r`, so you
can retune the rank without retuning the learning rate.

### Why this exists alongside problem 26
Interview sandboxes ship `jax` but not `flax`. Same class name, same argument
names, same `linear`/`lora_A`/`lora_B`/`scaling` attributes.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


class Linear:
    """Given to you, as nnx.Linear is in problem 26."""

    def __init__(self, d_in, d_out, *, key):
        self.kernel = jax.random.normal(key, (d_in, d_out)) / jnp.sqrt(d_in)
        self.bias = jnp.zeros((d_out,))

    def __call__(self, x):
        return x @ self.kernel + self.bias


class LoRALinear:
    """A frozen base projection plus a low-rank trainable adapter."""

    def __init__(self, in_features, out_features, rank, alpha=1.0, *, key):
        pass  # Replace this

    def __call__(self, x):
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

lora = LoRALinear(8, 4, rank=2, alpha=4.0, key=jax.random.key(0))
x = jax.random.normal(jax.random.key(1), (3, 8))

print("A", lora.lora_A.shape, " B", lora.lora_B.shape, " scaling", lora.scaling)
print("adapter is a no-op at init:", bool(jnp.allclose(lora(x), lora.linear(x))))

lora.lora_B = jnp.ones((2, 4)) * 0.1
print("after training B:          ", bool(jnp.allclose(lora(x), lora.linear(x))))

# "Frozen" is just a choice of what to differentiate.
def loss(A, B):
    lora.lora_A, lora.lora_B = A, B
    return jnp.sum(lora(x))

full = 8 * 4
adapter = 2 * (8 + 4)
print(f"\nparams: base {full}, adapter {adapter} ({adapter / full:.0%})")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("lora_pure")

# hint("lora_pure")      # stuck? nudge without the answer
# solution("lora_pure")  # spoiler: the reference implementation
# status()               # your dashboard across all problems